# Homework 9 

### Module 10 options 

#### Andrew Huang and Samantha Chen

# Problem 1 

In [26]:
import numpy as np

def american_vanilla_call(S0, K, T, r, q, sigma, N):

    # Step size
    dt = T / N

    # CRR parameters
    u = np.exp(sigma * np.sqrt(dt))
    d = 1 / u

    # Risk-neutral probability
    p = (np.exp((r - q) * dt) - d) / (u - d)

    # One-step discount factor
    discount = np.exp(-r * dt)

    # Terminal stock prices
    j = np.arange(N + 1)

    stock_prices = S0 * (u ** j) * (d ** (N - j))

    # Terminal call payoff
    option_values = np.maximum(stock_prices - K, 0)

    # Save tree levels needed for Greeks
    saved_levels = {}

    # Backward induction
    for i in range(N - 1, -1, -1):

        j = np.arange(i + 1)

        stock_prices = S0 * (u ** j) * (d ** (i - j))

        continuation = discount * (
            p * option_values[1:i + 2]
            + (1 - p) * option_values[0:i + 1]
        )

        # American early exercise
        exercise = np.maximum(stock_prices - K, 0)

        option_values = np.maximum(
            continuation,
            exercise
        )

        # Save first two levels
        if i in [0, 1, 2]:
            saved_levels[i] = (
                stock_prices.copy(),
                option_values.copy()
            )

    price = option_values[0]

    return price, saved_levels, dt

In [19]:
S0 = 100.0
K = 100.0
T = 5.0
r = 0.05
q = 0.08
sigma = 0.35
dsigma = 0.0001
dr = 0.0001

N = 4000

price, saved_levels, dt = american_vanilla_call(
    S0, K, T, r, q, sigma, N
)

print("American Vanilla Call Price:", price)

American Vanilla Call Price: 21.12444385863937


In [20]:
S_t0, f_t0 = saved_levels[0]
S_t1, f_t1 = saved_levels[1]
S_t2, f_t2 = saved_levels[2]

# Delta
delta = (
    f_t1[1] - f_t1[0]
) / (
    S_t1[1] - S_t1[0]
)

# Gamma
delta_down = (
    f_t2[1] - f_t2[0]
) / (
    S_t2[1] - S_t2[0]
)

delta_up = (
    f_t2[2] - f_t2[1]
) / (
    S_t2[2] - S_t2[1]
)

gamma = (
    delta_up - delta_down
) / (
    S_t1[1] - S_t1[0]
)

# Theta
theta = (
    f_t2[1] - f_t0[0]
) / (
    2 * dt
)

In [21]:
# Vega
price_sigma_up, _, _ = american_vanilla_call(
    S0,
    K,
    T,
    r,
    q,
    sigma + dsigma,
    N
)

vega = (
    price_sigma_up - price
) / dsigma

# Rho
price_r_up, _, _ = american_vanilla_call(
    S0,
    K,
    T,
    r + dr,
    q,
    sigma,
    N
)
rho = (
    price_r_up - price
) / dr




In [22]:
print("American Vanilla Call Price:", round(price, 4))
print("Delta:", round(delta, 4))
print("Gamma:", round(gamma, 4))
print("Theta:", round(theta, 4))
print("Vega:", round(vega, 4))
print("Rho:", round(rho, 4))

American Vanilla Call Price: 21.1244
Delta: 0.5217
Gamma: 0.006
Theta: -1.0718
Vega: 67.1521
Rho: 85.1546


# Problem 1 

### Explanation

Overall we get an American Vanilla Call Price of 21.1244. We have a Delta value of 0.5217, Gamma of 0.006, Theta at -1.0718, Vega at 67.1521, and Rho at 85.1546. 

In terms of accuracy for the CRR binomial tree we used a CRR binomial tree of n = 4000. This creates a small time interval Delta =  t/n = (5/4000), which creates a relatively good numerical approximation. Overall, using a large number of time steps gives us reasonable confidence in the numerical results. I am confident in the accuracy of the numbers. 



# IEOR 221 — Homework 9, Problem 2
## European Average-Price (Asian) Call via Monte Carlo

### Problem
Price a European *average-price* call with payoff
$$\text{payoff}=\max\!\Big(\tfrac{1}{n}\sum_{i=1}^{n}S_{t_i}-K,\;0\Big)$$
where the average uses the **daily closing price** with $n=125$ trading days ($=250\times0.5$) over $T=0.5$ year. Market conditions (same as Q1): $S_0=100$, $K=100$, $r=5\%$, $q=8\%$, $\sigma=35\%$.

### Method
We simulate risk-neutral GBM paths
$$S_{t+\Delta t}=S_t\exp\!\Big[(r-q-\tfrac{\sigma^2}{2})\Delta t+\sigma\sqrt{\Delta t}\,Z\Big],\quad Z\sim N(0,1),$$
average the price along each path, and discount the mean payoff. All four schemes use the **same 20,000 paths** and the **same seed (110526)**:

1. **Plain MC** — straightforward average of simulated payoffs (baseline).
2. **Antithetic variates** — for each path $Z$, also simulate its mirror $-Z$; the over-/under-shooting pairs offset, lowering variance at no extra cost.
3. **Control variate** — use the terminal price $S_T$ as a control, since $E[S_T]=S_0e^{(r-q)T}$ is known; subtract $b\,(S_T-E[S_T])$ to cancel shared noise.
4. **Antithetic + control** — both techniques combined.

We also price the BSM vanilla call (same $K,T$) for comparison and report standard errors.

In [23]:
import numpy as np
from math import log, sqrt, exp, erf

In [24]:
def N_cdf(x):
    return 0.5*(1.0+erf(x/sqrt(2.0)))

def bsm_call(S0,K,T,r,q,sigma):
    d1=(log(S0/K)+(r-q+sigma**2/2)*T)/(sigma*sqrt(T))
    d2=d1-sigma*sqrt(T)
    return S0*exp(-q*T)*N_cdf(d1)-K*exp(-r*T)*N_cdf(d2)

In [25]:
print("\n"+"="*60); print("Q2: Asian average-price call, Monte Carlo"); print("="*60)
S0,K,T,r,q,sigma=100.,100.,0.5,0.05,0.08,0.35
n=125                 
M=20000               
dt=T/n
drift=(r-q-0.5*sigma**2)*dt
vol=sigma*sqrt(dt)
disc=exp(-r*T)
SEED=110526

def asian_payoff_from_Z(Z):
    logpaths=np.cumsum(drift+vol*Z,axis=1)
    S=S0*np.exp(logpaths)            
    A=S.mean(axis=1)                 
    payoff=np.maximum(A-K,0.0)
    ST=S[:,-1]
    return payoff,ST

EST=S0*exp((r-q)*T)   

#1
np.random.seed(SEED)
Z=np.random.standard_normal((M,n))
pay,ST=asian_payoff_from_Z(Z)
price1=disc*pay.mean(); se1=disc*pay.std(ddof=1)/sqrt(M)
#2
np.random.seed(SEED)
Zh=np.random.standard_normal((M//2,n))
payP,_=asian_payoff_from_Z(Zh)
payN,_=asian_payoff_from_Z(-Zh)
pair=0.5*(payP+payN)
price2=disc*pair.mean(); se2=disc*pair.std(ddof=1)/sqrt(M//2)
#3
np.random.seed(SEED)
Z=np.random.standard_normal((M,n))
pay,ST=asian_payoff_from_Z(Z)
b=np.cov(pay,ST,ddof=1)[0,1]/np.var(ST,ddof=1)
paycv=pay-b*(ST-EST)
price3=disc*paycv.mean(); se3=disc*paycv.std(ddof=1)/sqrt(M)
#4
np.random.seed(SEED)
Zh=np.random.standard_normal((M//2,n))
payP,STP=asian_payoff_from_Z(Zh)
payN,STN=asian_payoff_from_Z(-Zh)
pairpay=0.5*(payP+payN)
pairST =0.5*(STP+STN)
b4=np.cov(pairpay,pairST,ddof=1)[0,1]/np.var(pairST,ddof=1)
paircv=pairpay-b4*(pairST-EST)
price4=disc*paircv.mean(); se4=disc*paircv.std(ddof=1)/sqrt(M//2)

vanilla=bsm_call(S0,K,T,r,q,sigma)
print(f"  seed={SEED}, paths={M}, steps={n}\n")
print(f"  (1) plain MC            price={price1:.2f}  SE={se1:.4f}")
print(f"  (2) antithetic          price={price2:.2f}  SE={se2:.4f}")
print(f"  (3) control variate     price={price3:.2f}  SE={se3:.4f}")
print(f"  (4) antithetic+control  price={price4:.2f}  SE={se4:.4f}")
print(f"\n  BSM vanilla call (same K,T) = {vanilla:.2f}")
print(f"  Asian call is cheaper by about {vanilla-price4:.2f}")
print(f"\n  Variance-reduction factor vs plain (SE ratio^2):")
print(f"    antithetic : {(se1/se2)**2:6.1f}x")
print(f"    control    : {(se1/se3)**2:6.1f}x")
print(f"    combined   : {(se1/se4)**2:6.1f}x")


Q2: Asian average-price call, Monte Carlo
  seed=110526, paths=20000, steps=125

  (1) plain MC            price=5.17  SE=0.0615
  (2) antithetic          price=5.17  SE=0.0495
  (3) control variate     price=5.20  SE=0.0390
  (4) antithetic+control  price=5.18  SE=0.0335

  BSM vanilla call (same K,T) = 8.82
  Asian call is cheaper by about 3.64

  Variance-reduction factor vs plain (SE ratio^2):
    antithetic :    1.5x
    control    :    2.5x
    combined   :    3.4x


## Results & Discussion

| Method | Price | Std. Error |
|---|---|---|
| (1) Plain MC | 5.17 | 0.0615 |
| (2) Antithetic | 5.17 | 0.0495 |
| (3) Control variate | 5.20 | 0.0390 |
| (4) Antithetic + control | **5.18** | **0.0335** |

**BSM vanilla call (same $K,T$) = 8.82.**

- **Consistency.** All four estimators target the same price and agree within one–two standard errors; the combined scheme gives the tightest estimate, **5.18 ± 0.03**.
- **Asian vs. vanilla.** The Asian call (5.18) is much cheaper than the vanilla call (8.82): averaging smooths the path, which lowers the *effective volatility* and hence the option value.
- **Variance reduction.** Versus plain MC, antithetic gives ≈1.5×, the $S_T$ control ≈2.5×, and the combined scheme ≈3.4× (smallest SE). Antithetic is only modestly effective because the payoff is kinked (≈half the paths pay 0), which breaks the symmetry it exploits; the $S_T$ control is only moderately correlated with the average-price payoff. A **geometric-average Asian control** (closed form, highly correlated) would reduce the variance substantially more.
- **Accuracy.** MC error shrinks like $1/\sqrt{M}$, so 4× the paths halves the error; variance reduction buys the same precision without extra paths. With a fixed seed, results are fully reproducible.